<a href="https://colab.research.google.com/github/dJasawat/Guvi_Assignmnets5_AI-Powered-Banking-Support-Fraud-Intelligence-System-using-NLP-RAG/blob/main/AI_Banking_RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install sentence-transformers groq qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 930.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 3.7 MB/s eta 0:00:00


In [ ]:
import os

In [ ]:
import os

if not os.path.isdir("RAG_Data"):
    from google.colab import files
    uploaded = files.upload()          # select RAG_Data.zip
    !unzip -o -q RAG_Data.zip

print("Files in RAG_Data/:")
print(sorted(os.listdir("RAG_Data")))

Saving RAG_Data.zip to RAG_Data.zip
Files in RAG_Data/:
['04_qa_pairs.json', 'fraud_handling_policy.txt', 'kyc_policy.txt', 'loan_processing_policy.txt', 'refund_dispute_policy.txt']


In [ ]:
import getpass

os.environ["GROQ_API_KEY"]= getpass.getpass("Paste your Groq key: ")
print("Key saved for this session ✔")

Paste your Groq key: ··········
Key saved for this session ✔


In [ ]:
from groq import Groq
from sentence_transformers import SentenceTransformer

# it reads groq key automatically
groq_client = Groq()

GROQ_MODEL="openai/gpt-oss-120b"

# Downloads the model the first time (~90 MB), then it's cached
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Connected to Groq and embedding model loaded ✔")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Connected to Groq and embedding model loaded ✔


In [ ]:
from pathlib import Path

RAG_Data_Folder= Path("RAG_Data")
documents={}

for file in sorted(RAG_Data_Folder.glob("*.txt")):
  documents[file.name]=file.read_text(encoding="utf-8")

print(f"Loaded {len(documents)} documents:\n")
for name,text in documents.items():
  print(f"{name:35s} {len(text):>7,} characters")


Loaded 0 documents:



In [ ]:
# split Texts
def split_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    text = " ".join((text or "").split())  #it's a text-cleaning operation. It removes unnecessary whitespace from a string and replaces multiple spaces/newlines/tabs with a single space.

    if not text:
        return []
    if chunk_size < 50:
        raise ValueError("Chunk size should be at least 100 characters.")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("Overlap must be zero or smaller than chunk size.")

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]

        if end < len(text):
            preferred_break = max(
                chunk.rfind(". "),
                chunk.rfind("? "),
                chunk.rfind("! "),
                chunk.rfind("\n"),
            )
            if preferred_break >= int(chunk_size * 0.55):
                end = start + preferred_break + 1
                chunk = text[start:end]

        chunk = chunk.strip()
        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = max(end - overlap, start + 1)

    return chunks


In [ ]:
all_chunks=[]

for filename,text in documents.items():
  for i,chunk in enumerate(split_text(text)):
    all_chunks.append(
        {"id":f"{filename}_chunk_{i}",
         "text":chunk,
         "source":filename,
        })
print(f"Total chunks across all documents: {len(all_chunks)}")
print("\nExample chunk:")
print(all_chunks[5])

Total chunks across all documents: 0

Example chunk:


IndexError: list index out of range

In [ ]:
#Text Embeddings
import numpy as np
import pandas as pd

def embed_texts(texts: list[str]) -> list[list[float]]:
    cleaned = [(text or "").strip() for text in texts]
    if not cleaned or any(not text for text in cleaned):
        raise ValueError("Every text must contain content.")
    return [list(vector) for vector in embedding_model.encode(cleaned)]

vectors = embed_texts([c["text"] for c in all_chunks])
vectors= np.array(vectors).astype("float32")

dimension = vectors.shape[1]

In [ ]:
print(f"Vector length: {len(vectors)}")
print(f"Vector dimension: {dimension}")
print(f"First 8 numbers: {[[round(x,4) for x in v[:5]] for v in vectors[:8]]}")

Vector length: 49
Vector dimension: 384
First 8 numbers: [[np.float32(-0.0478), np.float32(0.031), np.float32(-0.0178), np.float32(-0.0787), np.float32(0.024)], [np.float32(-0.0009), np.float32(0.0535), np.float32(-0.0214), np.float32(0.0107), np.float32(0.0741)], [np.float32(-0.063), np.float32(0.0393), np.float32(-0.0101), np.float32(-0.0766), np.float32(0.0223)], [np.float32(-0.0105), np.float32(0.027), np.float32(-0.0598), np.float32(-0.08), np.float32(-0.0358)], [np.float32(-0.022), np.float32(0.0494), np.float32(-0.0512), np.float32(-0.0323), np.float32(-0.0303)], [np.float32(-0.1159), np.float32(0.0241), np.float32(-0.0425), np.float32(-0.0414), np.float32(0.0145)], [np.float32(-0.1553), np.float32(0.0288), np.float32(0.054), np.float32(-0.0281), np.float32(0.0496)], [np.float32(-0.0159), np.float32(0.0329), np.float32(-0.0475), np.float32(0.0025), np.float32(0.024)]]


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sample_query = "What is fraud policy?"

# Embed the sample query
query_vector = embedding_model.encode([sample_query])[0].reshape(1, -1)

# Calculate cosine similarity between the query vector and all chunk vectors
# 'vectors' contains the embeddings of all chunks, each row is a chunk's embedding
similarity_scores = cosine_similarity(query_vector, vectors)

# The result is a 2D array, we need the first (and only) row
similarity_scores = similarity_scores[0]

# Calculate the average similarity score
average_similarity = similarity_scores.mean()

print(f"Average similarity score for all chunks with the query '{sample_query}': {average_similarity:.4f}")

# Optionally, you can also see the min and max scores for context
print(f"Minimum similarity score: {similarity_scores.min():.4f}")
print(f"Maximum similarity score: {similarity_scores.max():.4f}")

Average similarity score for all chunks with the query 'What is fraud policy?': 0.2608
Minimum similarity score: 0.0209
Maximum similarity score: 0.6965


In [ ]:

from qdrant_client import QdrantClient, models
os.environ["QDRANT_URL"] = "https://f2562d30-a491-4781-b7b1-0fe3bec4f6fd.sa-east-1-0.aws.cloud.qdrant.io"
os.environ["QDRANT_API_KEY"] = getpass.getpass("Paste your Qdrant key: ")
print("Key saved for this session ✔")

Paste your Qdrant key: ··········
Key saved for this session ✔


In [ ]:
client=QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"]
)
COLLECTION_NAME="RAG_Data"

In [ ]:
#Method Check If collection exists in Qdrant if not create Collection
def collection_exists(collection_name,vectorsDimensions) -> bool:
    try:
        # Check if collection exists, create if not
        if not client.collection_exists(collection_name=collection_name):
            client.create_collection(
                collection_name=collection_name,
                vectors_config=models.VectorParams(
                                        size=vectorsDimensions,
                                        distance=models.Distance.COSINE),
            )
            print(f"Collection '{collection_name}' created.")
        else:
            print(f"Collection '{collection_name}' already exists.")

    except Exception as e:
        print(f"Error creating or checking collection '{collection_name}': {e}")
        return

In [ ]:
def save_KnowledgeBase_to_qdrant(qdrant_url: str, qdrant_api_key: str, collection_name: str, vectors: np.ndarray, all_chunks: list):
    collection_exists(collection_name,vectors.shape[1])

    # Prepare points for upsert
    points = []
    for i, vector in enumerate(vectors):
        payload = {
            "text": all_chunks[i]["text"],
            "source": all_chunks[i]["source"],
            "id": all_chunks[i]["id"]
        }
        points.append(
            models.PointStruct(
                id=i,
                vector=vector.tolist(), # Qdrant expects list, not np.ndarray
                payload=payload
            )
        )

    try:
        # Upsert points
        operation_info = client.upsert(
            collection_name=collection_name,
            wait=True,
            points=points,
        )
        print(f"Upsert operation info: {operation_info}")
        print(f"Successfully saved {len(points)} embeddings to Qdrant collection '{collection_name}'.")

    except Exception as e:
        print(f"Error upserting vectors to collection '{collection_name}': {e}")

In [ ]:
# Save embedings in quadrants
#collection_name = "RAG_Data"
save_KnowledgeBase_to_qdrant(os.environ["QDRANT_URL"], os.environ["QDRANT_API_KEY"], COLLECTION_NAME, vectors, all_chunks)


Collection 'RAG_Data' already exists.
Upsert operation info: operation_id=2 status=<UpdateStatus.COMPLETED: 'completed'>
Successfully saved 49 embeddings to Qdrant collection 'RAG_Data'.


# Save Question and Answers in **Quadrant**

In [ ]:
QA_PAIR_COLLECTION="QA_Pairs"

In [ ]:
# Generate QA Pair text , encode them  and save into quadrant
import pandas as pd
qaDoc= pd.read_json("/content/RAG_Data/04_qa_pairs.json")


# Extract questions and answers
qa_text=[]
for _, row in qaDoc.iterrows():
    text = f"""Category : {row['category']}
               Question: {row['question']}
               Answer: {row['answer']}"""
    qa_text.append(text)


# Encode the questions and answers
print("\nEncoding questions and Answers...")
qa_vectors = embedding_model.encode(qa_text)

# Store encoded QA pairs with their original data
points = []
for i, row in qaDoc.iterrows():
    payload = {
        "id": row["id"],
        "category": row["category"],
        "question": row["question"],
        "answer": row["answer"],
        "suggested_action": row["suggested_action"],
         }

    points.append(
        models.PointStruct(
            id=i,
            vector=qa_vectors[i].tolist(), # Convert numpy array to list for storage
            payload=payload
        )
    )


try:
       #Check if collection exists
  collection_exists(QA_PAIR_COLLECTION,qa_vectors.shape[1])

        # save QA_payload in qdrant
  operation_info = client.upsert(
           collection_name=QA_PAIR_COLLECTION,
            wait=True,
            points=points )
  print(f"Upsert operation info: {operation_info}")
  print(f"Successfully saved {len(points)} embeddings to Qdrant collection '{QA_PAIR_COLLECTION}'.")

except Exception as e:
        print(f"Error upserting vectors to collection '{QA_PAIR_COLLECTION}': {e}")




Encoding questions and Answers...
Collection 'QA_Pairs' created.
Upsert operation info: operation_id=1 status=<UpdateStatus.COMPLETED: 'completed'>
Successfully saved 20 embeddings to Qdrant collection 'QA_Pairs'.


### Search Function for Qdrant

This function will embed a given query and use it to search the Qdrant collection for the most relevant documents.

In [ ]:
def search_qdrant(query: str, collection_name: str, embedding_model, client, top_k: int = 3) -> list[dict]:
    # Embed the query
    query_vector = embedding_model.encode([query])[0].tolist()

    # Perform the search in Knowledge Base
    search_result = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        with_payload=True # Include payload in the results
    )

    # Extract relevant information from the results
    formatted_results = []
    for point in search_result.points:
        payload = point.payload or {}
        formatted_results.append({
            "score": point.score,
            "text": payload.get("text", ""),
            "source": payload.get("source", ""),
            "id": payload.get("id", "")
        })

    #perform search in QA Pair
    qa_search_result = client.query_points(
        collection_name=QA_PAIR_COLLECTION,
        query=query_vector,
        limit=top_k,
        with_payload=True # Include payload in the results
    )
    qa_Formatted_results = []
    #extract relevent infor from the results
    for point in qa_search_result.points:
        payload = point.payload or {}
        qa_Formatted_results.append({
            "score": point.score,
            "Category": payload.get("category", ""),
            "suggested action": payload.get("suggested_action", ""),
            "answer": payload.get("answer", "")})



    return [formatted_results , qa_Formatted_results]

In [ ]:
def format_recent_history(history: list, max_messages: int = 6) -> str:
    if not history:
        return "No earlier conversation."

    lines = []
    for item in history[-max_messages:]:
        if isinstance(item, dict):
            role = item.get("role", "user")
            content = item.get("content", "")
            lines.append(f"{role.title()}: {content}")
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            lines.append(f"User: {item[0]}")
            lines.append(f"Assistant: {item[1]}")

    return "\n".join(lines) or "No earlier conversation."

def rag_chat(
    message: str,
    history: list,
    qdrant_url: str,
    qdrant_api_key: str,
    collection: str,
    groq_api_key: str,
    model_name: str,
    top_k: int,
    embedding_model, # New parameter
    client,          # New parameter
):
    history = list(history or [])

    try:
        message = (message or "").strip()
        if not message:
            return "", history

        # Corrected call to search_qdrant
        contexts = search_qdrant(
            message,
            collection,
            embedding_model, # Passed as parameter
            client,          # Passed as parameter
            top_k,
        )

        if not contexts:
            answer = "I could not find relevant information in the stored documents."
        elif not (groq_api_key or "").strip():
            answer = (
                "Retrieval succeeded, but no Groq key was provided. "
                "Here are the retrieved chunks:\n\n"
                + "\n\n".join(contexts)
            )
        else:
            # Extract text from each context dictionary for joining
            context_texts = [c["text"] for c in contexts[0]]
            context_texts.extend([c["answer"] for c in contexts[1]])
            category = [c["Category"] for c in contexts[1]]
            suggested_action = [c["suggested action"] for c in contexts[1]]

            # Buid Prompt
            prompt = f"""
            You are a careful retrieval-augmented assistant.

            Answer the current question using only the supplied context.
            Use recent conversation only to understand pronouns or follow-up questions.
            Do not introduce facts that are absent from the context.
            When the context does not contain the answer, say:
            "I could not find this information in the uploaded documents."


             Formatting rules:
            - Give a clear, concise answer.
            - Use Markdown formatting.
            - Use a short heading when appropriate.
            - Use bullet points for multiple items.
            - Use numbered lists for procedures or steps.
            - Use **bold** only for important terms.
            - Keep paragraphs short.
            - Do not repeat the conversation history.
            - Do not mention information that is not supported by the context.
            - Put sources at the end of the answer.
            - Do not use HTML.


            Current question:{message}

            Query Category:{category}

            Suggested Action:{suggested_action}

            Recent conversation:{format_recent_history(history)}

            Retrieved context:{chr(10).join(context_texts)}""".strip()

            # Call LLM
            groq_client = Groq(api_key=groq_api_key.strip())
            completion = groq_client.chat.completions.create(
                model=(model_name or "openai/gpt-oss-20b").strip(),
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You answer document questions accurately and "
                            "refuse to invent unsupported information."
                        ),
                    },
                    {"role": "user", "content": prompt},
                ],
                temperature=0.2,
                max_completion_tokens=1200,
            )
            answer = completion.choices[0].message.content

        history.extend(
            [
                {"role": "user", "content": message},
                {"role": "assistant", "content": answer},
            ]
        )
        return answer

    except Exception as exc:
        answer = f"❌ {exc}"
        history.extend(
            [
                {"role": "user", "content": message},
                {"role": "assistant", "content": answer},
            ]
        )
        return  answer

def clear_chat():
    return []



# **Predict intent of the Query - NLP Sentiment anlaysis**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
import os

module_path = "/content/drive/MyDrive/Guvi Projects/Guvi_Project5_NLP+RAG BankingSystem"
sys.path.append(module_path)

# --- Debugging additions ---
print(f"Checking module path: {module_path}")
if not os.path.exists(module_path):
    print(f"Error: The specified module directory does not exist: {module_path}")
else:
    print(f"Contents of {module_path}:")
    print(os.listdir(module_path))

    expected_file = "AI_Banking_NLP_functions.py"
    if expected_file not in os.listdir(module_path):
        print(f"Error: '{expected_file}' not found in '{module_path}'. Please ensure the file is there and the name is correct (case-sensitive).")
    else:
        print(f"'{expected_file}' found in '{module_path}'. Attempting import...")
# --------------------------



Checking module path: /content/drive/MyDrive/Guvi Projects/Guvi_Project5_NLP+RAG BankingSystem
Contents of /content/drive/MyDrive/Guvi Projects/Guvi_Project5_NLP+RAG BankingSystem:
['RawData', 'ProcessedData', 'Notebooks', 'models', '.ipynb_checkpoints', '__pycache__', 'AI_Banking_NLP_functions.py']
'AI_Banking_NLP_functions.py' found in '/content/drive/MyDrive/Guvi Projects/Guvi_Project5_NLP+RAG BankingSystem'. Attempting import...


In [ ]:
import importlib
import AI_Banking_NLP_functions
importlib.reload(AI_Banking_NLP_functions)

# Re-import the functions to use the reloaded module
from AI_Banking_NLP_functions import clean_data, predict_intent

print("AI_Banking_NLP_functions module reloaded and functions re-imported.")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


AI_Banking_NLP_functions module reloaded and functions re-imported.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
# Now, re-run the predict_intent function after the module reload
x,y = predict_intent("Multiple small transactions are showing up that I didn't make")

print(x,y)

Neutral {'Angry': 0.07769761847752606, 'Anxious': 0.16017248206463333, 'Confused': 0.17844301544651178, 'Frustrated': 0.14859000145929976, 'Neutral': 0.2743679231941759, 'Urgent': 0.16072895935785303}


In [ ]:

rag_chat("What is the fraud handling policy?","RAG_Data",
         os.environ["QDRANT_URL"],
         os.environ["QDRANT_API_KEY"],
         COLLECTION_NAME,
         os.environ["GROQ_API_KEY"],
         GROQ_MODEL,
         3,
         embedding_model, # Passed as parameter
         client
         )

'### Fraud Handling Policy (Retail Banking)\n\n- **Scope** – Covers all unauthorized transactions, card fraud, UPI fraud, net‑banking fraud, and identity‑theft cases reported by retail banking customers.【1†L1-L3】\n\n- **Key Procedures**\n  1. **Immediate actions** – Account access is suspended for safety; the fraud team begins investigation.  \n  2. **Regulatory compliance** –  \n     - RBI Master Direction (2017) on limiting customer liability applies.  \n     - Any fraud >\u202f₹1\u202fcrore must be reported to RBI within 24\u202fhours.  \n     - Quarterly fraud MIS is submitted to the Board Risk Committee.  \n  3. **Investigation timeline** – Completed within 30‑45 working days, with updates to the customer every 7 days.  \n  4. **Provisional credit** – May be applied within 10 working days for eligible cases.  \n\n- **Liability** – If the OTP was voluntarily shared, liability may apply per RBI guidelines; if obtained via phishing/vishing, the bank investigates and may waive liabili

In [ ]:
import os
import gradio as gr
import traceback

def process_user_query(chat_history, user_query_text):
    try:
        qdrant_url = os.environ.get("QDRANT_URL")
        qdrant_api_key = os.environ.get("QDRANT_API_KEY")
        groq_api_key = os.environ.get("GROQ_API_KEY")

          # Previous conversation for the LLM
        rag_history = [
                      message
                      for message in chat_history[:-1]
                      if message.get("content") != "Thinking..."
                     ]
        rag_answer = rag_chat(
            user_query_text,
            rag_history,
            qdrant_url,
            qdrant_api_key,
            COLLECTION_NAME,
            groq_api_key,
            GROQ_MODEL,
            3,
            embedding_model,
            client
        )

        # Perform Sentiment Analysis
        sentiment_prediction, sentiment_confidence = predict_intent(user_query_text)
        sentiment_details = ", ".join([f"{label}: {score:.2%}" for label, score in sentiment_confidence.items()])
        sentiment_output_str = f"**Sentiment Analysis:**\nPredicted Sentiment: {sentiment_prediction}\nConfidence: {sentiment_details}"

        # Update the placeholder in list-of-lists format
        chat_history[-1]["content"] = rag_answer

        return chat_history, sentiment_output_str

    except Exception as e:
        print("\n--- GRADIO DEBUG ERROR ---")
        traceback.print_exc()
        error_msg = f"❌ An internal error occurred: {e}"
        chat_history[-1][1] = error_msg
        return chat_history, "**Sentiment Analysis:**\nError: Could not analyze sentiment."

with gr.Blocks() as demo:
    gr.Markdown("# Banking RAG Chatbot with Sentiment Analysis")

    # Removed type="messages" to maintain compatibility
    chatbot = gr.Chatbot(label="Chat History", height=400)
    # lines=1 ensures Enter key triggers 'submit'
    msg_input = gr.Textbox(
        label="Your Query",
        placeholder="Type here and press Enter...",
        lines=1
    )

    # Correct way to add examples in Gradio Blocks
    gr.Examples(
        examples=[
            "I see a transaction of 500 I didn't make",
            "How do I update my KYC documents?",
            "My education loan status is still pending.",
            "I can't log into my net banking account.",
            "I need to change my address associated with my account.",
            "My debit card was stolen, what should I do?",
            "What are the requirements for opening a new savings account?"
        ],
        inputs=msg_input
    )

    sentiment_output_display = gr.Markdown(label="Sentiment Analysis", value="**Sentiment Analysis:**\nN/A")
    clear_btn = gr.Button("Clear")

    user_message_storage = gr.State("")

    def add_user_message(msg, history):
        if history is None:
            history = []
        # Add user msg and assistant placeholder in list-of-lists format
       # history.append([msg, "Thinking..."])
       # add history in dictionary format
        history.append({
            "role": "user",
            "content": msg
        })

        history.append({
            "role": "assistant",
            "content": "Thinking..."
        })

        return history, "", msg

    msg_input.submit(
        add_user_message,
        [msg_input, chatbot],
        [chatbot, msg_input, user_message_storage],
        queue=False
    ).then(
        process_user_query,
        [chatbot, user_message_storage],
        [chatbot, sentiment_output_display]
    )

    clear_btn.click(lambda: ([], "", "**Sentiment Analysis:**\nN/A"), None, [chatbot, msg_input, sentiment_output_display])

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://063466c28f313dd69f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Origional Code
import os
import gradio as gr
import traceback

def process_user_query(chat_history, user_query_text):
    try:
        qdrant_url = os.environ.get("QDRANT_URL")
        qdrant_api_key = os.environ.get("QDRANT_API_KEY")
        groq_api_key = os.environ.get("GROQ_API_KEY")

          # Previous conversation for the LLM
        rag_history = [
                      message
                      for message in chat_history[:-1]
                      if message.get("content") != "Thinking..."
                     ]
        rag_answer = rag_chat(
            user_query_text,
            rag_history,
            qdrant_url,
            qdrant_api_key,
            COLLECTION_NAME,
            groq_api_key,
            GROQ_MODEL,
            3,
            embedding_model,
            client
        )

        # Perform Sentiment Analysis
        sentiment_prediction, sentiment_confidence = predict_intent(user_query_text)
        sentiment_details = ", ".join([f"{label}: {score:.2%}" for label, score in sentiment_confidence.items()])
        sentiment_output_str = f"**Sentiment Analysis:**\nPredicted Sentiment: {sentiment_prediction}\nConfidence: {sentiment_details}"

        # Update the placeholder in list-of-lists format
        chat_history[-1]["content"] = rag_answer

        return chat_history, sentiment_output_str

    except Exception as e:
        print("\n--- GRADIO DEBUG ERROR ---")
        traceback.print_exc()
        error_msg = f"❌ An internal error occurred: {e}"
        chat_history[-1][1] = error_msg
        return chat_history, "**Sentiment Analysis:**\nError: Could not analyze sentiment."

with gr.Blocks() as demo:
    gr.Markdown("# Banking RAG Chatbot with Sentiment Analysis")

    # Removed type="messages" to maintain compatibility
    chatbot = gr.Chatbot(label="Chat History", height=400)
    # lines=1 ensures Enter key triggers 'submit'
    msg_input = gr.Textbox(label="Your Query", placeholder="Type here and press Enter...", lines=1)
    sentiment_output_display = gr.Markdown(label="Sentiment Analysis", value="**Sentiment Analysis:**\nN/A")
    clear_btn = gr.Button("Clear")

    user_message_storage = gr.State("")

    def add_user_message(msg, history):
        if history is None:
            history = []
        # Add user msg and assistant placeholder in list-of-lists format
       # history.append([msg, "Thinking..."])
       # add history in dictionary format
        history.append({
            "role": "user",
            "content": msg
        })

        history.append({
            "role": "assistant",
            "content": "Thinking..."
        })

        return history, "", msg

    msg_input.submit(
        add_user_message,
        [msg_input, chatbot],
        [chatbot, msg_input, user_message_storage],
        queue=False
    ).then(
        process_user_query,
        [chatbot, user_message_storage],
        [chatbot, sentiment_output_display]
    )

    clear_btn.click(lambda: ([], "", "**Sentiment Analysis:**\nN/A"), None, [chatbot, msg_input, sentiment_output_display])

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6d88fbc867fe18e080.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Testing purpose only
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle
from sklearn.preprocessing import LabelEncoder
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import os

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
DIR_PATH = "/content/drive/MyDrive/Guvi Projects/Guvi_Project5_NLP+RAG BankingSystem/"

# code to get TDFvectorizer from the drive
tfidf_vectorizer_path = os.path.join(DIR_PATH,"models","tfidf_vectorizer_sentiment.pkl")
with open(tfidf_vectorizer_path, 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
#tfidf_vectorizer = TfidfVectorizer(max_features=200)


def clean_data(text):
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenize the text
    tokens = nltk.word_tokenize(text)
    # Remove stop words
    tokens = [word for word in tokens if word not in stop_words]
    # Lemmatization
    lemmatizer = nltk.WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)


def predict_intent(user_query):
    # 1. Preprocess the input text
    cleaned_text = clean_data(user_query)

    # 2. Vectorize the cleaned text
    # Note: tfidf_vectorizer and tuned_models must be defined in previous cells
    vectorized_text = tfidf_vectorizer.transform([cleaned_text])

    # 3. Predict using the Logistic Regression model (highest performer)
    model_path =  os.path.join(DIR_PATH,"models","logistic_regression_model.pkl")

    with open(model_path, 'rb') as f:
         model = pickle.load(f)

    prediction = model.predict(vectorized_text)[0]

    # 4. Get confidence/probabilities
    probabilities = model.predict_proba(vectorized_text)[0]
    labels = model.classes_
    conf_dict = {labels[i]: float(probabilities[i]) for i in range(len(labels))}

    return prediction, conf_dict



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


### Demo of the Search Function

Let's test the search function with a sample query.

In [ ]:
sample_query = "What are the steps for processing a loan?"
search_results = search_qdrant(sample_query, collection_name="RAG_Data", embedding_model=embedding_model, top_k=3)

print(f"Search results for query: '{sample_query}'\n")
for i, result in enumerate(search_results):
    print(f"--- Result {i+1} (Score: {result['score']:.4f}) ---")
    print(f"Source: {result['source']}")
    print(f"ID: {result['id']}")
    print(f"Text: {result['text']}\n")

NameError: name 'QA_PAIR_COLLECTION' is not defined

In [ ]:


# Define Gradio Interface
interface = gr.Interface(
    fn=predict_intent,
    inputs=gr.Textbox(lines=2, placeholder="Enter your banking query here...", label="Support Ticket Text"),
    outputs=[
        gr.Textbox(label="Predicted Category"),
        gr.Label(label="Confidence Scores")
    ],
    title="Banking Intent Classifier",
    description="Identify the category of a support ticket (e.g., KYC, Loan, Fraud, Account Access) based on the query text.",
    examples=[
        ["I see a transaction of 500 I didn't make"],
        ["How do I update my KYC documents?"],
        ["My education loan status is still pending."],
        ["I can't log into my net banking account."],
        ["I need to change my address associated with my account."],
        ["My debit card was stolen, what should I do?"],
        ["What are the requirements for opening a new savings account?"]
    ]
)


NameError: name 'gr' is not defined